# Anomaly Classifier
## AeroNet Lite - Module 5 (Part B)
**BS Data Science AI Semester Project - SP2026**

This notebook covers Part B of the ML Pipeline: training classifiers to detect
flight anomalies from synthetic drone telemetry. The output feeds into the
disruption handler (Module 4) and the main simulation loop.

---
### What this notebook does
1. Generates synthetic drone telemetry with four anomaly classes
2. Explores feature distributions per class
3. Trains Decision Tree, Random Forest, KNN, and Naive Bayes classifiers
4. Evaluates with accuracy, confusion matrix, and cross-validation
5. Exposes `detect_anomaly()` for teammate integration

### Integration points exposed to teammates
| Function | Used by | Purpose |
|---|---|---|
| `detect_anomaly(battery_drop, speed, route_deviation, altitude_change, speed_change)` | Module 4 (Disruption Handler), main.py | Returns anomaly class string for each telemetry reading |


## 0. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13

print('All libraries loaded.')
print('anomaly_classifier.ipynb - Module 5 Part B')


## B1. Generate Synthetic Telemetry Dataset

Real UAV telemetry is not publicly available in a clean, labeled form suitable for a 2-week
project. We generate synthetic flight logs following the anomaly rules defined in the project
specification. The rules are clearly stated below so they can be explained during viva.

| Class | Rule |
|---|---|
| Normal | Battery drops gradually, route deviation low |
| Battery_Anomaly | Battery drop unusually high |
| Route_Anomaly | Route deviation unusually high |
| Sensor_Spike | Altitude or speed change unusually large |


In [ ]:
N_PER_CLASS = 500


def make_normal(n):
    return pd.DataFrame({
        'battery_drop'    : np.random.normal(5, 3, n).clip(0, 15),
        'speed'           : np.random.normal(15, 4, n).clip(3, 30),
        'route_deviation' : np.random.normal(2, 1.5, n).clip(0, 8),
        'altitude_change' : np.random.normal(0, 2, n).clip(-6, 6),
        'speed_change'    : np.random.normal(0, 2, n).clip(-7, 7),
        'label'           : 'Normal'
    })


def make_battery_anomaly(n):
    return pd.DataFrame({
        'battery_drop'    : np.random.normal(12, 5, n).clip(2, 25),
        'speed'           : np.random.normal(13, 4, n).clip(3, 28),
        'route_deviation' : np.random.normal(3, 2, n).clip(0, 10),
        'altitude_change' : np.random.normal(1, 2.5, n).clip(-6, 8),
        'speed_change'    : np.random.normal(1, 2.5, n).clip(-7, 9),
        'label'           : 'Battery_Anomaly'
    })


def make_route_anomaly(n):
    return pd.DataFrame({
        'battery_drop'    : np.random.normal(6, 3, n).clip(0, 16),
        'speed'           : np.random.normal(19, 5, n).clip(3, 33),
        'route_deviation' : np.random.normal(8, 4, n).clip(1, 20),
        'altitude_change' : np.random.normal(1, 2.5, n).clip(-6, 8),
        'speed_change'    : np.random.normal(2, 3, n).clip(-7, 10),
        'label'           : 'Route_Anomaly'
    })


def make_sensor_spike(n):
    return pd.DataFrame({
        'battery_drop'    : np.random.normal(6, 3.5, n).clip(0, 16),
        'speed'           : np.random.normal(16, 5, n).clip(3, 32),
        'route_deviation' : np.random.normal(3, 2.5, n).clip(0, 11),
        'altitude_change' : np.random.normal(10, 6, n).clip(-2, 28),
        'speed_change'    : np.random.normal(12, 7, n).clip(-3, 35),
        'label'           : 'Sensor_Spike'
    })


telemetry = pd.concat([
    make_normal(N_PER_CLASS),
    make_battery_anomaly(N_PER_CLASS),
    make_route_anomaly(N_PER_CLASS),
    make_sensor_spike(N_PER_CLASS)
], ignore_index=True)

telemetry = telemetry.sample(frac=1, random_state=42).reset_index(drop=True)

print('Shape:', telemetry.shape)
print('\nClass distribution:')
print(telemetry['label'].value_counts())


In [ ]:
print(telemetry.columns.tolist())
print()
print(telemetry.head())


## B2. Telemetry Exploratory Data Analysis

In [ ]:
TELE_FEATURES = ['battery_drop', 'speed', 'route_deviation', 'altitude_change', 'speed_change']

fig, axes = plt.subplots(1, len(TELE_FEATURES), figsize=(18, 4))

for ax, feat in zip(axes, TELE_FEATURES):
    for label in telemetry['label'].unique():
        subset = telemetry[telemetry['label'] == label][feat]
        ax.hist(subset, bins=30, alpha=0.5, label=label)
    ax.set_title(feat)
    ax.set_xlabel('Value')
    ax.legend(fontsize=7)

plt.suptitle('Feature Distributions by Anomaly Class', fontsize=13)
plt.tight_layout()
plt.show()


## B3. Prepare Classification Data

In [ ]:
le_label = LabelEncoder()
telemetry['label_enc'] = le_label.fit_transform(telemetry['label'])

print('Class encoding:')
for cls, enc in zip(le_label.classes_, range(len(le_label.classes_))):
    print(f'  {enc} -> {cls}')

X_tel = telemetry[TELE_FEATURES]
y_tel = telemetry['label_enc']

X_tr_t, X_te_t, y_tr_t, y_te_t = train_test_split(
    X_tel, y_tel, test_size=0.2, random_state=42, stratify=y_tel
)

print(f'\nTraining samples : {X_tr_t.shape[0]}')
print(f'Test samples     : {X_te_t.shape[0]}')


## B4. Train Multiple Classifiers

In [ ]:
classifiers = {
    'Decision Tree' : DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest' : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'KNN'           : KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes'   : GaussianNB()
}

results = {}

for name, clf in classifiers.items():
    clf.fit(X_tr_t, y_tr_t)
    y_pred = clf.predict(X_te_t)
    acc = accuracy_score(y_te_t, y_pred)
    results[name] = {'model': clf, 'predictions': y_pred, 'accuracy': acc}
    print(f'{name:20s}  Accuracy: {acc:.4f}')


## B5. Detailed Evaluation of Best Classifier

In [ ]:
best_clf_name   = max(results, key=lambda k: results[k]['accuracy'])
best_clf_result = results[best_clf_name]
best_clf        = best_clf_result['model']
y_pred_best     = best_clf_result['predictions']

print(f'Best classifier : {best_clf_name}')
print(f'Accuracy        : {best_clf_result["accuracy"]:.4f}')
print()
print('Classification Report:')
print(classification_report(y_te_t, y_pred_best, target_names=le_label.classes_))


## B6. Confusion Matrices for All Classifiers

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
class_names = le_label.classes_

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_te_t, res['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f'{name}\nAcc={res["accuracy"]:.3f}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.tick_params(axis='x', rotation=30)
    ax.tick_params(axis='y', rotation=0)

plt.suptitle('Confusion Matrices - Anomaly Detection Classifiers', fontsize=13)
plt.tight_layout()
plt.show()


## B7. Classifier Accuracy Comparison

In [ ]:
names = list(results.keys())
accs  = [results[n]['accuracy'] for n in names]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(names, accs, color=['#4C72B0', '#DD8452', '#55A868', '#C44E52'])
ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy')
ax.set_title('Classifier Accuracy Comparison - Anomaly Detection')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, acc + 0.01,
            f'{acc:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

print(f'Best classifier : {best_clf_name} with accuracy {max(accs):.4f}')


## B8. Cross-Validation of Best Classifier

In [ ]:
cv_scores = cross_val_score(best_clf, X_tel, y_tel, cv=5, scoring='accuracy')

print(f'Cross-validation results for {best_clf_name}:')
print(f'  Per-fold accuracies : {[round(s, 4) for s in cv_scores]}')
print(f'  Mean accuracy       : {cv_scores.mean():.4f}')
print(f'  Std deviation       : {cv_scores.std():.4f}')
print()
print('A low standard deviation means the model generalises well across data splits.')


## B9. Integration Function - detect_anomaly()

In [ ]:
def detect_anomaly(battery_drop, speed, route_deviation, altitude_change, speed_change):
    """
    Classify a drone telemetry reading as Normal or one of three anomaly types.

    Parameters
    ----------
    battery_drop     : float - Battery percentage dropped in this step
    speed            : float - Current drone speed (m/s or grid units per step)
    route_deviation  : float - Cells deviated from planned route
    altitude_change  : float - Change in altitude since last step
    speed_change     : float - Change in speed since last step

    Returns
    -------
    str : One of 'Normal', 'Battery_Anomaly', 'Route_Anomaly', 'Sensor_Spike'
    """
    features = np.array([[battery_drop, speed, route_deviation, altitude_change, speed_change]])
    pred_enc = best_clf.predict(features)[0]
    return le_label.inverse_transform([pred_enc])[0]


print('detect_anomaly() smoke tests:')
print(f'  Normal flight        : {detect_anomaly(2.0,  15.0, 0.5,  0.1,  0.5)}')
print(f'  Battery drop         : {detect_anomaly(30.0, 14.0, 1.0,  0.0,  0.5)}')
print(f'  Far from route       : {detect_anomaly(3.0,  16.0, 15.0, 0.5,  1.0)}')
print(f'  Altitude+speed spike : {detect_anomaly(2.0,  15.0, 0.5,  30.0, 40.0)}')


## B10. Integration with Other Modules

This section shows exactly how teammates call `detect_anomaly()`.
**No changes to this notebook are required after initial run.**


In [ ]:
# -----------------------------------------------------------------------
# HOW MODULE 4 (Disruption Handler) calls this notebook
# -----------------------------------------------------------------------
# In delivery_simulator.py or main.py:
#
#   from ml_pipeline import detect_anomaly
#
#   anomaly = detect_anomaly(
#       battery_drop=drone.battery_drop,
#       speed=drone.speed,
#       route_deviation=drone.route_deviation,
#       altitude_change=drone.altitude_change,
#       speed_change=drone.speed_change
#   )
#   if anomaly != 'Normal':
#       disruption_handler.trigger(drone_id=drone.id, anomaly_type=anomaly)

print('--- Module 4 (Disruption Handler) integration example ---')

# Simulate a telemetry reading from Drone D3 at step 18
drone_d3_telemetry = {
    'battery_drop'    : 35.0,
    'speed'           : 13.0,
    'route_deviation' : 1.0,
    'altitude_change' : 0.5,
    'speed_change'    : 1.0
}

anomaly_type = detect_anomaly(**drone_d3_telemetry)
print(f'  Drone D3 telemetry  : {drone_d3_telemetry}')
print(f'  Detected anomaly    : {anomaly_type}')
if anomaly_type != 'Normal':
    print(f'  Action              : Disruption handler triggers return-to-hub for Drone D3.')


In [ ]:
# -----------------------------------------------------------------------
# HOW main.py calls this at simulation steps 18-19
# -----------------------------------------------------------------------
print('--- main.py simulation steps 18-19 ---')

# Step 18 - Anomaly injection
print('Step 18: Telemetry received for Drone D3.')
anomaly_18 = detect_anomaly(
    battery_drop=35, speed=13, route_deviation=1.0,
    altitude_change=0.5, speed_change=1.0
)
print(f'         Anomaly detection result: {anomaly_18}')
if anomaly_18 != 'Normal':
    print(f'         ALERT: {anomaly_18} detected on Drone D3. Initiating return-to-hub protocol.')

# Step 19 - Reroute decision
print()
print('Step 19: Disruption handler evaluates reroute or hub return.')
if anomaly_18 == 'Battery_Anomaly':
    print('         Battery issue confirmed. Drone D3 returning to nearest hub.')
elif anomaly_18 == 'Route_Anomaly':
    print('         Route deviation confirmed. Recalculating path via A* from current position.')
elif anomaly_18 == 'Sensor_Spike':
    print('         Sensor spike detected. Grounding Drone D3 pending diagnostics.')
else:
    print('         No anomaly. Drone D3 continues on planned route.')


In [ ]:
# -----------------------------------------------------------------------
# HOW this notebook connects back to demand_forecasting.ipynb
# -----------------------------------------------------------------------
# If you run the full project, main.py imports from ml_pipeline.py which
# combines both notebooks into one importable module.
#
# Execution order in main.py:
#   1. demand_forecasting outputs -> grid cell demand values (Steps 15-16)
#   2. anomaly_classifier outputs -> flight safety decisions (Steps 18-19)
#
# Both are used in the 20-step simulation. They do not depend on each other
# directly; they share only the common grid model (grid_model.py).

print('--- Combined simulation flow overview ---')
print('Steps 15-16 : demand_forecasting.ipynb -> get_demand_forecast()')
print('Step 17     : anomaly_classifier.ipynb -> cross_val_score printed here')
print(f'  Cross-val mean accuracy: {cv_scores.mean():.4f}')
print('Steps 18-19 : anomaly_classifier.ipynb -> detect_anomaly()')
print('Step 20     : main.py prints final summary')


## Summary

In [ ]:
print('ANOMALY CLASSIFIER NOTEBOOK - SUMMARY')
print('=' * 55)
print(f'Dataset      : Synthetic telemetry ({len(telemetry)} samples, 4 classes)')
print(f'Classes      : {list(le_label.classes_)}')
print(f'Features     : {TELE_FEATURES}')
print(f'Best model   : {best_clf_name}')
print(f'Accuracy     : {best_clf_result["accuracy"]:.4f}')
print(f'CV Mean Acc  : {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')
print()
print('Integration export:')
print('  detect_anomaly(battery_drop, speed, route_deviation,')
print('                 altitude_change, speed_change) -> str')
print()
print('Simulation steps handled here: 17 (cross-val), 18 (detection), 19 (action)')
print('=' * 55)
